# k-Fold Cross Validation (COUGHVID)

使用 Train 資料集（專家標註，已 balanced）做 k-fold cross validation。

> 改為 **5-fold**（原本 8-fold），每個 test fold 樣本數更多，結果更穩定。

## Imports

In [1]:
import os
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import classification_report
import torch
from utils import CoughNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

c:\Users\aint\anaconda3\envs\base1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Hyperparameters

In [2]:
hparams = {
    'dataset': 'data/prepared_train_coughvid_balanced.csv',
    'epochs': 20,
    'batch_size': 16,
    'lr': 1e-3,
    'features': [
        'chroma_stft', 'rmse', 'spectral_centroid', 'spectral_bandwidth', 'rolloff', 'zero_crossing_rate',
        'mfcc1', 'mfcc2', 'mfcc3', 'mfcc4', 'mfcc5', 'mfcc6', 'mfcc7', 'mfcc8', 'mfcc9', 'mfcc10',
        'mfcc11', 'mfcc12', 'mfcc13', 'mfcc14', 'mfcc15', 'mfcc16', 'mfcc17', 'mfcc18', 'mfcc19', 'mfcc20'
    ]
}

## Prepare Data

In [3]:
df_features = pd.read_csv(hparams['dataset'])
X = np.array(df_features[hparams['features']], dtype=np.float32)

encoder = LabelEncoder()
y = encoder.fit_transform(df_features['label'])
print('classes:', encoder.classes_)
print('label 分佈:')
print(df_features['label'].value_counts())
print('總樣本數:', len(df_features))

classes: ['covid' 'healthy' 'symptomatic']
label 分佈:
label
covid          470
healthy        470
symptomatic    470
Name: count, dtype: int64
總樣本數: 1410


## 5-Fold Cross Validation

In [4]:
k_folds = 5
kfold   = KFold(n_splits=k_folds, shuffle=True, random_state=42)
indices = np.arange(len(y))
results_train = []
results_test  = []

criterion = torch.nn.CrossEntropyLoss()

def train(loader_train, model, optimizer, epoch):
    model.train()
    running_correct = 0.0
    total = 0
    for batch_ndx, sample in enumerate(loader_train):
        features, labels = sample[0].to(device), sample[1].to(device)
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        predictions = torch.argmax(outputs.data, 1)
        running_correct += (predictions == labels).sum().item()
        total += labels.shape[0]
    return running_correct / total

def evaluate(loader_test, model, epoch):
    model.eval()
    running_correct = 0.0
    total = 0
    all_preds  = []
    all_labels = []
    with torch.no_grad():
        for batch_ndx, sample in enumerate(loader_test):
            features, labels = sample[0].to(device), sample[1].to(device)
            outputs = model(features)
            predictions = torch.argmax(outputs.data, 1)
            running_correct += (predictions == labels).sum().item()
            total += labels.shape[0]
            all_preds.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return running_correct / total, all_preds, all_labels

print(f'K-FOLD CROSS VALIDATION RESULTS FOR {k_folds} FOLDS')
print('--------------------------------------------')
print('|         | Train Accuracy | Test Accuracy |')
print('--------------------------------------------')

last_preds  = None
last_labels = None

for fold, (train_ids, test_ids) in enumerate(kfold.split(indices)):
    X_train, X_test = X[train_ids], X[test_ids]
    y_train, y_test = y[train_ids], y[test_ids]

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    torch.manual_seed(42)
    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.Tensor(X_train), torch.Tensor(y_train).long()),
        batch_size=hparams['batch_size'], shuffle=True)
    test_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.Tensor(X_test), torch.Tensor(y_test).long()),
        batch_size=hparams['batch_size'], shuffle=False)

    model     = CoughNet(len(hparams['features'])).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=hparams['lr'])

    for epoch in range(hparams['epochs']):
        train_accuracy = train(train_loader, model, optimizer, epoch)
        eval_accuracy, preds, labels = evaluate(test_loader, model, epoch)

    results_train.append(train_accuracy)
    results_test.append(eval_accuracy)
    last_preds  = preds
    last_labels = labels
    print(f'| Fold {fold}  |       {train_accuracy*100:.2f} % |       {eval_accuracy*100:.2f} % |')

print('--------------------------------------------')
print(f'| Average |       {np.mean(results_train)*100:.2f} % |       {np.mean(results_test)*100:.2f} % |')
print(f'| Std Dev |       {np.std(results_train)*100:.2f} % |       {np.std(results_test)*100:.2f} % |')

K-FOLD CROSS VALIDATION RESULTS FOR 5 FOLDS
--------------------------------------------
|         | Train Accuracy | Test Accuracy |
--------------------------------------------
| Fold 0  |       84.75 % |       33.69 % |
| Fold 1  |       89.10 % |       36.52 % |
| Fold 2  |       86.35 % |       40.43 % |
| Fold 3  |       90.34 % |       39.72 % |
| Fold 4  |       87.85 % |       37.94 % |
--------------------------------------------
| Average |       87.68 % |       37.66 % |
| Std Dev |       1.97 % |       2.41 % |


## Classification Report（最後一個 Fold）

In [5]:
print(classification_report(last_labels, last_preds, target_names=encoder.classes_))

              precision    recall  f1-score   support

       covid       0.38      0.38      0.38        89
     healthy       0.42      0.42      0.42        98
 symptomatic       0.33      0.34      0.34        95

    accuracy                           0.38       282
   macro avg       0.38      0.38      0.38       282
weighted avg       0.38      0.38      0.38       282

